In [49]:
from langchain_community.document_loaders import WebBaseLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
import os

In [37]:
from dotenv import load_dotenv

In [38]:
def load_data(url):
    loader = WebBaseLoader(url)
    docs = loader.load()
    return docs
docs = load_data("https://brainlox.com/courses/category/technical")
print(docs[0].page_content[:1000]) 

Brainlox: Learn technical courses.Courses TechnicalAcademicLanguageMusicLifestyleBook a Free Demo NowSign InFAQContact UsHomeCoursesCoursesWe found great courses available for you$30per sessionLEARN SCRATCH PROGRAMING
Scratch Course is the foundation of coding and is a building block of a coding journey. If you want 16 LessonsView Details$30per sessionLEARN CLOUD COMPUTING BASICS-AWS
In this course we are going to cover the basics and the most important services on AWS,
At the end  20 LessonsView Details$30per sessionLEARN MOBILE DEVELOPMENT
Mobile application development is the process of creating software applications that run on a mobil 24 LessonsView Details$30per sessionLEARN CORE JAVA PROGRAMMING ONLINE
Java is a very popular high-level, class-based, object-oriented programming language that is design 41 LessonsView Details$30per sessionLEARN ROBOTICS
You can open all kinds of doors for advancement in so many careers with a basic understanding of el 25 LessonsView Details$30per s

In [40]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain.embeddings import HuggingFaceEmbeddings

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
documents = text_splitter.split_documents(docs)

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
print(embeddings)


C:\Users\athar\AppData\Local\Temp\ipykernel_13348\2292434972.py:8: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


client=SentenceTransformer(
  (0): Transformer({'max_seq_length': 256, 'do_lower_case': False}) with Transformer model: BertModel 
  (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
) model_name='sentence-transformers/all-MiniLM-L6-v2' cache_folder=None model_kwargs={} encode_kwargs={} multi_process=False show_progress=False


In [54]:
print(documents)

[Document(metadata={'source': 'https://brainlox.com/courses/category/technical', 'title': 'Brainlox: Learn technical courses.', 'description': 'Your one stop education destination!', 'language': 'zxx'}, page_content='Brainlox: Learn technical courses.Courses TechnicalAcademicLanguageMusicLifestyleBook a Free Demo NowSign InFAQContact UsHomeCoursesCoursesWe found great courses available for you$30per sessionLEARN SCRATCH PROGRAMING\nScratch Course is the foundation of coding and is a building block of a coding journey. If you want 16 LessonsView Details$30per sessionLEARN CLOUD COMPUTING BASICS-AWS\nIn this course we are going to cover the basics and the most important services on AWS,'), Document(metadata={'source': 'https://brainlox.com/courses/category/technical', 'title': 'Brainlox: Learn technical courses.', 'description': 'Your one stop education destination!', 'language': 'zxx'}, page_content='At the end  20 LessonsView Details$30per sessionLEARN MOBILE DEVELOPMENT\nMobile applic

In [52]:
vectors = embeddings.embed_query("What are yo doing?")
len(vectors)

384

In [83]:
from pinecone import Pinecone
import os
os.environ["PINECONE_API_KEY"] = "pcsk_748Zfb_HciVT3heHenUizcDLrZfDZX5pj9F51bzqmCfy1M7LFY7uvyxLxFbmfSF1WNTiot"


pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])

index_name = "langchain"
index = pc.Index(index_name)
print(pc.list_indexes()) 

{'indexes': [{'deletion_protection': 'disabled',
              'dimension': 384,
              'host': 'langchain-u1s46cl.svc.aped-4627-b74a.pinecone.io',
              'metric': 'cosine',
              'name': 'langchain',
              'spec': {'serverless': {'cloud': 'aws', 'region': 'us-east-1'}},
              'status': {'ready': True, 'state': 'Ready'}}]}


In [84]:

from langchain.vectorstores import Pinecone

vectorstore = Pinecone.from_documents(
    documents=documents, 
    embedding=embeddings, 
    index_name=index_name
)

In [85]:
def retrieve(query,k=2):
    res = vectorstore.similarity_search(query=query,k = k)
    return res


In [93]:
from langchain.chains.question_answering import load_qa_chain
from langchain_groq import ChatGroq
import os

groq_api_key = "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"

llm = ChatGroq(model_name="llama3-8b-8192", api_key=groq_api_key)

chain = load_qa_chain(llm,chain_type="stuff")


C:\Users\athar\AppData\Local\Temp\ipykernel_13348\947139891.py:18: LangChainDeprecationWarning: This class is deprecated. See the following migration guides for replacements based on `chain_type`:
stuff: https://python.langchain.com/docs/versions/migrating_chains/stuff_docs_chain
map_reduce: https://python.langchain.com/docs/versions/migrating_chains/map_reduce_chain
refine: https://python.langchain.com/docs/versions/migrating_chains/refine_chain
map_rerank: https://python.langchain.com/docs/versions/migrating_chains/map_rerank_docs_chain

See also guides on retrieval and question-answering here: https://python.langchain.com/docs/how_to/#qa-with-rag
  chain = load_qa_chain(llm,chain_type="stuff")


In [94]:
def retriever_ans(query):
    web_search = retrieve(query)
    print(web_search)
    res = chain.run(input_documents=web_search,question=query)
    return res




In [96]:
our_query = "what are the different aws course?"
ans = retriever_ans(our_query)
print(ans)

[Document(metadata={'description': 'Your one stop education destination!', 'language': 'zxx', 'source': 'https://brainlox.com/courses/category/technical', 'title': 'Brainlox: Learn technical courses.'}, page_content='Brainlox: Learn technical courses.Courses TechnicalAcademicLanguageMusicLifestyleBook a Free Demo NowSign InFAQContact UsHomeCoursesCoursesWe found great courses available for you$30per sessionLEARN SCRATCH PROGRAMING\nScratch Course is the foundation of coding and is a building block of a coding journey. If you want 16 LessonsView Details$30per sessionLEARN CLOUD COMPUTING BASICS-AWS\nIn this course we are going to cover the basics and the most important services on AWS,'), Document(metadata={'description': 'Your one stop education destination!', 'language': 'zxx', 'source': 'https://brainlox.com/courses/category/technical', 'title': 'Brainlox: Learn technical courses.'}, page_content='This introduction to cloud computing on Amazon AWS course takes you from the AWS Ad 18 

In [99]:
from flask import Flask, request, jsonify
from flask_restful import Api, Resource

app = Flask(__name__)
api = Api(app)

class Chatbot(Resource):
    def post(self):
        data = request.get_json()
        query = data.get("query")

        if not query:
            return {"error": "Query is required"}, 400
        web_search = retrieve(query)
        response = chain.run(input_documents=web_search,question=query)
        return {"response": response}

api.add_resource(Chatbot, "/chat")

if __name__ == "__main__":
    app.run()


 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
127.0.0.1 - - [08/Feb/2025 01:58:15] "POST /chat HTTP/1.1" 400 -
127.0.0.1 - - [08/Feb/2025 01:59:43] "GET / HTTP/1.1" 404 -
127.0.0.1 - - [08/Feb/2025 01:59:46] "GET / HTTP/1.1" 404 -
127.0.0.1 - - [08/Feb/2025 01:59:56] "GET /doc HTTP/1.1" 404 -
